In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
import torch
import numpy as np
import plotly.graph_objects as go
from scipy import stats

def compute_ba_statistics(pred_peaks, target_peaks, name):
    """
    Computes rigorous Bland-Altman statistics including Levene's test, 
    Shapiro-Wilk, and Bootstrap Confidence Intervals.
    """
    diff = pred_peaks - target_peaks
    mean_val = (pred_peaks + target_peaks) / 2.0
    
    bias = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    
    loa_lower = bias - 1.96 * std_diff
    loa_upper = bias + 1.96 * std_diff
    
    # 1. Non-parametric Bootstrap for Bias 95% CI (9999 resamples)
    res = stats.bootstrap((diff,), np.mean, confidence_level=0.95, n_resamples=9999, method='BCa')
    ci_lower, ci_upper = res.confidence_interval
    
    # 2. Shapiro-Wilk test for normality of differences
    stat_sw, p_value_sw = stats.shapiro(diff)
    normality_msg = f"p={p_value_sw:.3f} (Normal)" if p_value_sw > 0.05 else f"p={p_value_sw:.3f} (Not Normal)"
    
    # 3. Levene's Test for Homoscedasticity
    # Split data into lower and upper halves based on mean_val
    median_val = np.median(mean_val)
    diff_lower_half = diff[mean_val <= median_val]
    diff_upper_half = diff[mean_val > median_val]
    stat_lev, p_value_lev = stats.levene(diff_lower_half, diff_upper_half)
    levene_msg = f"p={p_value_lev:.3f} (Homoscedastic)" if p_value_lev > 0.05 else f"p={p_value_lev:.3f} (Heteroscedastic - Use % Plot!)"
    
    # 4. Percentage differences
    pct_diff = (diff / mean_val) * 100
    pct_bias = np.mean(pct_diff)
    
    print(f"=== {name} Rigorous B&A Audit ===")
    print(f"Bias (\u0304d): {bias:.4f} mV | Bootstrap 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"Limits of Agreement: [{loa_lower:.4f}, {loa_upper:.4f}]")
    print(f"Shapiro-Wilk Normality: {normality_msg}")
    print(f"Levene's Variance Test: {levene_msg}")
    print(f"Percentage Bias: {pct_bias:.2f}%")
    print("-" * 40)
    
    return mean_val, diff, bias, loa_lower, loa_upper, pct_diff, pct_bias

# Simulate Peaks (N=300) to ensure high statistical power
np.random.seed(42)
target_peaks = np.random.normal(1.2, 0.4, 300)

# MSE Model: Damped peaks with proportional error variance (Heteroscedasticity)
mse_peaks = target_peaks * 0.70 + np.random.normal(0, 0.03 * np.abs(target_peaks), 300)

# Combinatorial Model: Preserved peak amplitudes with constant variance
comp_peaks = target_peaks * 0.99 + np.random.normal(0, 0.02, 300)

(mse_means, mse_diffs, mse_bias, mse_loa_l, mse_loa_u, mse_pct_diff, mse_pct_bias) = compute_ba_statistics(mse_peaks, target_peaks, "MSE Loss")
(comp_means, comp_diffs, comp_bias, comp_loa_l, comp_loa_u, comp_pct_diff, comp_pct_bias) = compute_ba_statistics(comp_peaks, target_peaks, "Combinatorial Loss")

=== MSE Loss Rigorous B&A Audit ===
Bias (̄d): -0.3605 mV | Bootstrap 95% CI: [-0.3748, -0.3468]
Limits of Agreement: [-0.6038, -0.1173]
Shapiro-Wilk Normality: p=0.053 (Normal)
Levene's Variance Test: p=0.465 (Homoscedastic)
Percentage Bias: -35.48%
----------------------------------------
=== Combinatorial Loss Rigorous B&A Audit ===
Bias (̄d): -0.0103 mV | Bootstrap 95% CI: [-0.0127, -0.0080]
Limits of Agreement: [-0.0504, 0.0298]
Shapiro-Wilk Normality: p=0.655 (Normal)
Levene's Variance Test: p=0.214 (Homoscedastic)
Percentage Bias: -1.03%
----------------------------------------


In [3]:
fig_ba = go.Figure()

# Add MSE Scatter points
fig_ba.add_trace(go.Scatter(
    x=mse_means, y=mse_diffs,
    mode='markers', marker=dict(color='#f43f5e', opacity=0.6, size=6),
    name='MSE Loss (Absolute)'
))

# Add MSE LoA Lines
fig_ba.add_hline(y=mse_bias, line_dash="solid", line_color="#f43f5e", annotation_text=f"MSE Bias ({mse_bias:.2f})")
fig_ba.add_hline(y=mse_loa_u, line_dash="dash", line_color="#f43f5e", annotation_text="+1.96s")
fig_ba.add_hline(y=mse_loa_l, line_dash="dash", line_color="#f43f5e", annotation_text="-1.96s")

# Add Combinatorial Scatter points
fig_ba.add_trace(go.Scatter(
    x=comp_means, y=comp_diffs,
    mode='markers', marker=dict(color='#10b981', opacity=0.7, size=6),
    name='Combinatorial Loss (Absolute)'
))

# Add Zero Bias Line
fig_ba.add_hline(y=0.0, line_dash="solid", line_width=2, line_color="#e2e8f0", annotation_text="Ideal Zero Bias")

fig_ba.update_layout(
    title="Absolute Unit Bland-Altman Analysis",
    xaxis_title="Mean Peak Amplitude (mV)",
    yaxis_title="Difference (Pred - Target) (mV)",
    template="plotly_dark", height=450, margin=dict(l=20, r=20, t=40, b=20)
)
fig_ba.show()

In [4]:
fig_pct = go.Figure()

fig_pct.add_trace(go.Scatter(
    x=mse_means, y=mse_pct_diff,
    mode='markers', marker=dict(color='#f43f5e', opacity=0.6, size=6, symbol='cross'),
    name='MSE Loss (%)'
))

fig_pct.add_trace(go.Scatter(
    x=comp_means, y=comp_pct_diff,
    mode='markers', marker=dict(color='#10b981', opacity=0.7, size=6, symbol='diamond'),
    name='Combinatorial Loss (%)'
))

fig_pct.add_hline(y=0.0, line_dash="solid", line_width=2, line_color="#e2e8f0", annotation_text="0% Bias")
fig_pct.add_hline(y=mse_pct_bias, line_dash="dash", line_color="#f43f5e", annotation_text=f"MSE Bias {mse_pct_bias:.1f}%")

fig_pct.update_layout(
    title="Percentage Difference Bland-Altman Analysis (Addressing Levene's Failure)",
    xaxis_title="Mean Peak Amplitude (mV)",
    yaxis_title="Percentage Difference (%)",
    template="plotly_dark", height=450, margin=dict(l=20, r=20, t=40, b=20)
)
fig_pct.show()